In [ ]:
from pathlib import Path
import kagglehub

DATA_DIR = Path(kagglehub.dataset_download("colinmorris/reddit-usernames"))

print("Dataset folder:", DATA_DIR)
print("\nFiles inside:")

for p in DATA_DIR.rglob("*"):
    if p.is_file():
        print(p, round(p.stat().st_size / 1_000_000, 2), "MB")

100%|██████████| 213M/213M [00:10<00:00, 20.9MB/s]

Extracting files...


Dataset folder: /root/.cache/kagglehub/datasets/colinmorris/reddit-usernames/versions/1

Files inside:
/root/.cache/kagglehub/datasets/colinmorris/reddit-usernames/versions/1/users.csv 376.24 MB


In [2]:
import pandas as pd
import numpy as np
import math
from pathlib import Path
import kagglehub
from collections import Counter, defaultdict

DATA_DIR = Path(kagglehub.dataset_download("colinmorris/reddit-usernames"))

print("Dataset folder:", DATA_DIR)

candidate_files = [
    p for p in DATA_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in [".csv", ".tsv", ".txt"]
]

if not candidate_files:
    raise FileNotFoundError("No CSV, TSV, or TXT files found inside the Kaggle dataset folder.")

print("\nCandidate files:")
for p in candidate_files:
    print(p, round(p.stat().st_size / 1_000_000, 2), "MB")

INPUT_FILE = max(candidate_files, key=lambda p: p.stat().st_size)

print("\nUsing actual file:", INPUT_FILE)
print("File size:", round(INPUT_FILE.stat().st_size / 1_000_000, 2), "MB")


CHUNK_SIZE = 500_000

USERNAME_COL = None

def detect_username_column(file_path):
    sample = pd.read_csv(file_path, nrows=1000)
    cols = list(sample.columns)

    preferred = ["username", "user_name", "user", "author", "name"]

    for pref in preferred:
        for col in cols:
            if col.lower() == pref:
                return col

    for col in cols:
        low = col.lower()
        if "username" in low or "author" in low or "user" in low:
            return col

    raise ValueError(
        f"Could not detect username column. Columns found: {cols}. "
        "Set USERNAME_COL manually."
    )


if USERNAME_COL is None:
    USERNAME_COL = detect_username_column(INPUT_FILE)

print(f"Using username column: {USERNAME_COL}")

length_hist = Counter()
total_n = 0
sum_len = 0
sum_len_sq = 0

for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):
    s = chunk[USERNAME_COL].dropna().astype(str).str.strip()
    s = s[s != ""]

    lengths = s.str.len().astype(int)

    total_n += len(lengths)
    sum_len += lengths.sum()
    sum_len_sq += (lengths ** 2).sum()

    length_hist.update(lengths.tolist())

mean_len = sum_len / total_n
std_len = math.sqrt((sum_len_sq - (sum_len ** 2 / total_n)) / (total_n - 1))

print("\nLength stats")
print("------------")
print(f"N usernames: {total_n:,}")
print(f"Mean length: {mean_len:.3f}")
print(f"Standard deviation: {std_len:.3f}")


def quantile_from_hist(hist, q):
    target = math.ceil(q * total_n)
    running = 0
    for length in sorted(hist):
        running += hist[length]
        if running >= target:
            return length
    return max(hist)


q1 = quantile_from_hist(length_hist, 0.25)
q2 = quantile_from_hist(length_hist, 0.50)
q3 = quantile_from_hist(length_hist, 0.75)

print(f"Q1 length: {q1}")
print(f"Median length: {q2}")
print(f"Q3 length: {q3}")


METRIC_SUM_COLUMNS = [
    "contains_digit",
    "ends_with_digit",
    "starts_with_letter",
    "contains_underscore",
    "contains_other_special",
    "all_lowercase",
    "all_uppercase",
    "mixed_case",
    "camelcase_pattern",
    "has_repeated_char_3plus",
    "digit_count",
    "letter_count",
    "vowel_count",
    "delimiter_piece_count",
]


def empty_agg():
    d = defaultdict(float)
    d["n"] = 0
    d["sum_length"] = 0
    d["sum_length_sq"] = 0
    return d


overall = empty_agg()
quartile_aggs = defaultdict(empty_agg)
sd_band_aggs = defaultdict(empty_agg)


def make_features(s):
    f = pd.DataFrame(index=s.index)

    f["length"] = s.str.len().astype(int)

    f["contains_digit"] = s.str.contains(r"\d", regex=True).astype(int)
    f["ends_with_digit"] = s.str.contains(r"\d$", regex=True).astype(int)
    f["starts_with_letter"] = s.str.contains(r"^[A-Za-z]", regex=True).astype(int)

    f["contains_underscore"] = s.str.contains("_", regex=False).astype(int)
    f["contains_other_special"] = s.str.contains(
        r"[^A-Za-z0-9_]", regex=True
    ).astype(int)

    f["all_lowercase"] = s.str.fullmatch(r"[a-z0-9_]+").fillna(False).astype(int)
    f["all_uppercase"] = s.str.fullmatch(r"[A-Z0-9_]+").fillna(False).astype(int)

    has_lower = s.str.contains(r"[a-z]", regex=True)
    has_upper = s.str.contains(r"[A-Z]", regex=True)

    f["mixed_case"] = (has_lower & has_upper).astype(int)
    f["camelcase_pattern"] = s.str.contains(
        r"[a-z][A-Z]|[A-Z][a-z]", regex=True
    ).astype(int)

    f["has_repeated_char_3plus"] = s.str.contains(
        r"(.)\1\1+", regex=True
    ).astype(int)

    f["digit_count"] = s.str.count(r"\d")
    f["letter_count"] = s.str.count(r"[A-Za-z]")
    f["vowel_count"] = s.str.count(r"[AEIOUaeiou]")

    f["delimiter_piece_count"] = s.str.count(r"[_\-.]+") + 1

    return f


def update_agg(agg, f):
    n = len(f)
    if n == 0:
        return

    agg["n"] += n
    agg["sum_length"] += f["length"].sum()
    agg["sum_length_sq"] += (f["length"] ** 2).sum()

    for col in METRIC_SUM_COLUMNS:
        agg[col] += f[col].sum()


def quartile_label(length):
    if length <= q1:
        return f"Q1_shortest_len<={q1}"
    elif length <= q2:
        return f"Q2_len_{q1+1}_to_{q2}"
    elif length <= q3:
        return f"Q3_len_{q2+1}_to_{q3}"
    else:
        return f"Q4_longest_len>{q3}"


def sd_band_label(length):
    low = mean_len - std_len
    high = mean_len + std_len

    if length < low:
        return f"below_1sd_len<{low:.2f}"
    elif length <= high:
        return f"within_1sd_len_{low:.2f}_to_{high:.2f}"
    else:
        return f"above_1sd_len>{high:.2f}"


for chunk in pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE):
    s = chunk[USERNAME_COL].dropna().astype(str).str.strip()
    s = s[s != ""]

    f = make_features(s)

    update_agg(overall, f)

    quartile_labels = f["length"].apply(quartile_label)
    for label, sub in f.groupby(quartile_labels):
        update_agg(quartile_aggs[label], sub)

    sd_labels = f["length"].apply(sd_band_label)
    for label, sub in f.groupby(sd_labels):
        update_agg(sd_band_aggs[label], sub)


def finalize_agg(name, agg):
    n = agg["n"]

    if n == 0:
        return None

    mean_length = agg["sum_length"] / n

    if n > 1:
        std_length = math.sqrt(
            (agg["sum_length_sq"] - (agg["sum_length"] ** 2 / n)) / (n - 1)
        )
    else:
        std_length = 0

    row = {
        "group": name,
        "n_usernames": int(n),

        "mean_length": mean_length,
        "std_length": std_length,

        "pct_contains_digit": 100 * agg["contains_digit"] / n,
        "pct_ends_with_digit": 100 * agg["ends_with_digit"] / n,
        "pct_starts_with_letter": 100 * agg["starts_with_letter"] / n,
        "pct_contains_underscore": 100 * agg["contains_underscore"] / n,
        "pct_contains_other_special": 100 * agg["contains_other_special"] / n,

        "pct_all_lowercase": 100 * agg["all_lowercase"] / n,
        "pct_all_uppercase": 100 * agg["all_uppercase"] / n,
        "pct_mixed_case": 100 * agg["mixed_case"] / n,
        "pct_camelcase_pattern": 100 * agg["camelcase_pattern"] / n,

        "pct_repeated_char_3plus": 100 * agg["has_repeated_char_3plus"] / n,

        "avg_digit_count": agg["digit_count"] / n,
        "avg_letter_count": agg["letter_count"] / n,
        "avg_vowel_count": agg["vowel_count"] / n,
        "avg_delimiter_piece_count": agg["delimiter_piece_count"] / n,
    }

    return row


overall_df = pd.DataFrame([finalize_agg("overall", overall)])

quartile_df = pd.DataFrame(
    [finalize_agg(k, v) for k, v in quartile_aggs.items()]
).sort_values("group")

sd_band_df = pd.DataFrame(
    [finalize_agg(k, v) for k, v in sd_band_aggs.items()]
).sort_values("group")

for df in [overall_df, quartile_df, sd_band_df]:
    numeric_cols = df.select_dtypes(include="number").columns
    df[numeric_cols] = df[numeric_cols].round(3)


overall_df.to_csv("reddit_username_metrics_overall.csv", index=False)
quartile_df.to_csv("reddit_username_metrics_by_length_quartile.csv", index=False)
sd_band_df.to_csv("reddit_username_metrics_by_sd_band.csv", index=False)

print("\nSaved files:")
print("reddit_username_metrics_overall.csv")
print("reddit_username_metrics_by_length_quartile.csv")
print("reddit_username_metrics_by_sd_band.csv")

print("\nOverall metrics:")
print(overall_df.T)

print("\nMetrics by length quartile:")
print(quartile_df)

print("\nMetrics by standard-deviation band:")
print(sd_band_df)

Dataset folder: /root/.cache/kagglehub/datasets/colinmorris/reddit-usernames/versions/1

Candidate files:
/root/.cache/kagglehub/datasets/colinmorris/reddit-usernames/versions/1/users.csv 376.24 MB

Using actual file: /root/.cache/kagglehub/datasets/colinmorris/reddit-usernames/versions/1/users.csv
File size: 376.24 MB
Using username column: author

Length stats
------------
N usernames: 25,865,739
Mean length: 11.017
Standard deviation: 3.684
Q1 length: 8
Median length: 11
Q3 length: 13


/tmp/ipykernel_1651/627632009.py:164: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f["has_repeated_char_3plus"] = s.str.contains(
/tmp/ipykernel_1651/627632009.py:164: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f["has_repeated_char_3plus"] = s.str.contains(
/tmp/ipykernel_1651/627632009.py:164: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f["has_repeated_char_3plus"] = s.str.contains(
/tmp/ipykernel_1651/627632009.py:164: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  f["has_repeated_char_3plus"] = s.str.contains(
/tmp/ipykernel_1651/627632009.py:164: UserWarning: This pattern is interpreted as a regular expression, and has match gr


Saved files:
reddit_username_metrics_overall.csv
reddit_username_metrics_by_length_quartile.csv
reddit_username_metrics_by_sd_band.csv

Overall metrics:
                                   0
group                        overall
n_usernames                 25865739
mean_length                   11.017
std_length                     3.684
pct_contains_digit            29.465
pct_ends_with_digit           24.902
pct_starts_with_letter        98.048
pct_contains_underscore       10.847
pct_contains_other_special     2.389
pct_all_lowercase              52.23
pct_all_uppercase              2.094
pct_mixed_case                44.812
pct_camelcase_pattern         44.373
pct_repeated_char_3plus         2.67
avg_digit_count                0.789
avg_letter_count              10.039
avg_vowel_count                3.622
avg_delimiter_piece_count      1.182

Metrics by length quartile:
                group  n_usernames  mean_length  std_length  \
0  Q1_shortest_len<=8      7353864        6.903    

In [3]:
print(overall_df.T)

                                   0
group                        overall
n_usernames                 25865739
mean_length                   11.017
std_length                     3.684
pct_contains_digit            29.465
pct_ends_with_digit           24.902
pct_starts_with_letter        98.048
pct_contains_underscore       10.847
pct_contains_other_special     2.389
pct_all_lowercase              52.23
pct_all_uppercase              2.094
pct_mixed_case                44.812
pct_camelcase_pattern         44.373
pct_repeated_char_3plus         2.67
avg_digit_count                0.789
avg_letter_count              10.039
avg_vowel_count                3.622
avg_delimiter_piece_count      1.182
